# Example - Closed-Loop Simulation with a Kalman Filter

A plant, a Kalman filter (state estimator) and a state-feedback controller
are connected in closed loop with `connect_systems`, and simulated.

The controller has direct transmission: its output u(k) depends on its
input at the same time k, the state estimates. Here
u(k) = 0.38 r(k) - [0.1, 0.3] x_est(k|k), with the estimates x_est(k|k) of
a Kalman filter in filtering form (`KFF`), which corrects them with the
noisy measurement y(k) + v(k).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from cas_models.discrete_time.kalman import KFF, KFP
from cas_models.discrete_time.models import StateSpaceModelDTFromABCD
from cas_models.discrete_time.simulate import (
    make_n_step_simulation_function_from_model,
)
from cas_models.transformations import connect_systems

## The plant, the filter and the controller

In [ ]:
A = np.array([[0.8, 0.5], [0.0, 0.6]])
B = np.array([[0.0], [1.0]])
C = np.array([[1.0, 0.0]])
D = np.zeros((1, 1))
plant = StateSpaceModelDTFromABCD(
    A, B, C, D, dt=1.0, name="plant", input_names=["u"], output_names=["y"]
)
Q = np.diag([0.01, 0.01])  # the filter's process noise covariance
R = np.array([[0.04]])  # and measurement noise covariance (std. 0.2)
observer = KFF(A, B, C, D, Q, R, dt=1.0, name="observer")
print(observer.input_names, observer.output_names)

In [ ]:
controller = StateSpaceModelDTFromABCD(
    np.zeros((0, 0)), np.zeros((0, 3)), np.zeros((1, 0)),
    [[0.38, -0.1, -0.3]],  # u = 0.38 r - 0.1 xh_1 - 0.3 xh_2
    dt=1.0, name="controller",
    input_names=["r", "xh_1", "xh_2"], output_names=["uc"],
)

## Connecting them

The connections set each subsystem input (the keys) from outputs or new
external inputs (the values): the controller's output `uc` drives the
plant and the filter's input `uk`; the filter's measurement input `yk` is
the sum of the plant's output `y` and the measurement noise `v`, a new
external input, listed in `input_names` with the reference `r`.

In [ ]:
connections = {
    "u": "uc",  # the plant's input
    "uk": "uc",  # the filter's inputs: u(k) and the measurement
    "yk": {"y": 1.0, "v": 1.0},
    "xh_1": "xk_est_1",  # the controller's inputs
    "xh_2": "xk_est_2",
}
system = connect_systems(
    [plant, observer, controller], connections, input_names=["r", "v"]
)
print(system.input_names, system.output_names)

At each time k the connected system computes its outputs in the order of
their dependencies: the plant's y(k) (from its states), the filter's
correction x_est(k|k) (from y(k)), then the controller's u(k). Only then are
the states updated, and the filter's prediction x_est(k+1|k) uses the u(k)
just computed. So the filter's correction and prediction steps don't have
to run one after the other, and there is no algebraic loop: the filter's
outputs don't depend on u(k) (D = 0).

A filter in prediction form (`KFP`) outputs x_est(k+1|k), which does depend
on u(k) (the prediction needs it), so the same connections form an
algebraic loop. Use a `KFP` with controllers that have a one-step delay
(e.g. MPC), or a `KFF`:

In [ ]:
try:
    connect_systems(
        [plant, KFP(A, B, C, D, Q, R, dt=1.0, name="observer"), controller],
        connections | {"xh_1": "xkp1_est_1", "xh_2": "xkp1_est_2"},
        input_names=["r", "v"],
    )
except ValueError as err:
    print(err)

## Simulation

A step in the reference, with measurement noise.

In [ ]:
nT = 40
simulate = make_n_step_simulation_function_from_model(system, nT)
t = np.arange(nT + 1) * 1.0
r = np.where(t >= 5, 1.0, 0.0)
v = np.random.default_rng(0).normal(scale=0.2, size=nT + 1)
x0 = np.r_[np.zeros(4), np.eye(2).ravel()]  # plant, x_est(0|-1), P(0|-1)
X, Y = simulate(t, np.c_[r, v][:-1], x0)
X, Y = np.array(X), np.array(Y)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(7, 5.5), sharex=True)

ax = axes[0]
ax.plot(t, Y[:, 0] + v, ".", color="tab:blue", alpha=0.5, label="y measured")
ax.plot(t, Y[:, 0], color="tab:blue", label="y")
ax.plot(t, r, ":", color="tab:blue", drawstyle="steps-post", label="r")
ax.set_ylabel("y")
ax.grid(True)
ax.legend(loc="upper left")
ax.set_title("Outputs")

ax = axes[1]
for i, color in enumerate(["tab:orange", "tab:green"]):
    ax.plot(t, X[:, i], color=color, label=f"x{i + 1}")
    ax.plot(t, Y[:, 1 + i], "--", color=color, label=f"x{i + 1} est.")
ax.set_ylabel("x")
ax.grid(True)
ax.legend(loc="upper left", ncol=2)
ax.set_title("States and estimates x_est(k|k)")

ax = axes[2]
ax.plot(t, Y[:, 3], drawstyle="steps-post", label="u")
ax.set_ylabel("u")
ax.set_xlabel("k")
ax.grid(True)
ax.set_title("Inputs")

plt.tight_layout()
plt.show()